# Railway Track Defect Detection - YOLOv8 Training
## Dataset: Railway Defect Detection (2,324 images, 15 classes)
### Broad Gauge (1676mm) - Pakistan Railways

## 1. Setup & Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Install Dependencies

In [ ]:
!pip install -q ultralytics roboflow
import ultralytics
ultralytics.checks()

## 3. Download Dataset from Roboflow
Replace `YOUR_API_KEY` with your actual Roboflow API key.

Dataset: **Railway Defect Detection** by Model Train NP

In [ ]:
from roboflow import Roboflow

rf = Roboflow(api_key="oAHPVi0KEDHQXipmAaqd")
project = rf.workspace("model-train-np").project("railway-defect-detection-xomw5")
version = project.version(1)
dataset = version.download("yolov8")

print(f"Dataset downloaded to: {dataset.location}")
print(f"Number of classes: {len(project.classes)}")
print(f"Classes: {project.classes}")

## 4. Dataset Exploration

In [ ]:
import os
import yaml

data_yaml_path = os.path.join(dataset.location, "data.yaml")
with open(data_yaml_path, 'r') as f:
    data = yaml.safe_load(f)
    print(yaml.dump(data, default_flow_style=False))

# Count images in each split
for split in ['train', 'val', 'test']:
    split_path = os.path.join(dataset.location, split)
    if os.path.exists(split_path):
        images = [f for f in os.listdir(split_path) if f.endswith(('.jpg', '.jpeg', '.png'))]
        print(f"{split}: {len(images)} images")

## 5. Visualize Sample Annotations

In [ ]:
import cv2
import matplotlib.pyplot as plt
import random
from glob import glob

def plot_random_samples(dataset_path, num_samples=5):
    img_files = glob(os.path.join(dataset_path, 'train', 'images', '*'))[:num_samples*5]
    samples = random.sample(img_files, min(num_samples, len(img_files)))
    
    fig, axes = plt.subplots(1, len(samples), figsize=(20, 5))
    if len(samples) == 1:
        axes = [axes]
    
    for ax, img_path in zip(axes, samples):
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        # Read corresponding label
        label_path = img_path.replace('images', 'labels').replace('.jpg', '.txt').replace('.png', '.txt')
        if os.path.exists(label_path):
            with open(label_path, 'r') as f:
                lines = f.readlines()
            h, w, _ = img.shape
            for line in lines:
                parts = line.strip().split()
                cls_id = int(parts[0])
                x_center, y_center, box_w, box_h = map(float, parts[1:5])
                x1 = int((x_center - box_w/2) * w)
                y1 = int((y_center - box_h/2) * h)
                x2 = int((x_center + box_w/2) * w)
                y2 = int((y_center + box_h/2) * h)
                cv2.rectangle(img, (x1, y1), (x2, y2), (255, 0, 0), 3)
                cv2.putText(img, project.classes[cls_id], (x1, y1-10),
                           cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 0), 2)
        
        ax.imshow(img)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

plot_random_samples(dataset.location, num_samples=4)

## 6. Train YOLOv8 Model

**Model**: YOLOv8m (medium) - best balance of speed & accuracy
**Epochs**: 120 (sufficient for 2.3k images)
**Image Size**: 640x640

In [ ]:
from ultralytics import YOLO
import os

# Create output directory in Google Drive
output_dir = '/content/drive/MyDrive/railway_defect_output'
os.makedirs(output_dir, exist_ok=True)

# Load base model
model = YOLO('yolov8m.pt')

# Train the model
results = model.train(
    data=data_yaml_path,
    epochs=120,
    imgsz=640,
    batch=16,
    patience=20,
    device='cuda',
    workers=8,
    optimizer='AdamW',
    lr0=0.001,
    lrf=0.01,
    momentum=0.937,
    weight_decay=0.0005,
    warmup_epochs=3,
    cos_lr=True,
    augment=True,
    save=True,
    save_period=10,
    project=output_dir,
    name='yolov8m_railway_defects',
    exist_ok=True,
    pretrained=True,
    verbose=True,
    plots=True
)

print(f"Training complete. Model saved to: {output_dir}/yolov8m_railway_defects")

## 7. Evaluate Model on Test Set

In [ ]:
# Load the best model
best_model_path = f'{output_dir}/yolov8m_railway_defects/weights/best.pt'
model = YOLO(best_model_path)

# Run validation
metrics = model.val(
    data=data_yaml_path,
    split='test',
    imgsz=640,
    batch=16,
    plots=True,
    save_json=True
)

print(f"mAP50: {metrics.box.map50:.4f}")
print(f"mAP50-95: {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall: {metrics.box.mr:.4f}")

# Per-class metrics
print("\nPer-class mAP50:")
for i, cls_name in enumerate(project.classes):
    if i < len(metrics.box.ap_class_index):
        idx = list(metrics.box.ap_class_index).index(i) if i in metrics.box.ap_class_index else -1
        if idx >= 0:
            ap50 = metrics.box.ap50[idx]
            print(f"  {cls_name}: {ap50:.4f}")

## 8. Run Inference on Test Images

In [ ]:
import glob

# Get test images
test_images = glob.glob(os.path.join(dataset.location, 'test', 'images', '*'))[:8]

# Run inference
results = model(test_images, conf=0.25, iou=0.45)

# Display results
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

for i, (result, ax) in enumerate(zip(results, axes)):
    img = result.plot()
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax.axis('off')

plt.tight_layout()
plt.show()

## 9. Export Model for Deployment

Export to ONNX format for use in the web app (FastAPI).

In [ ]:
# Export to ONNX
model.export(format='onnx', imgsz=640)
onnx_path = best_model_path.replace('.pt', '.onnx')
print(f"ONNX model saved to: {onnx_path}")

# Copy best.pt and best.onnx to a dedicated deployment folder
import shutil
deploy_dir = '/content/drive/MyDrive/railway_defect_deploy'
os.makedirs(deploy_dir, exist_ok=True)
shutil.copy(best_model_path, os.path.join(deploy_dir, 'best.pt'))
if os.path.exists(onnx_path):
    shutil.copy(onnx_path, os.path.join(deploy_dir, 'best.onnx'))

# Also copy class names
import json
with open(os.path.join(deploy_dir, 'class_names.json'), 'w') as f:
    json.dump(project.classes, f, indent=2)

print(f"Deployment files saved to: {deploy_dir}")
!ls -la "{deploy_dir}"

## 10. Download Model to Local Machine

Run this cell to zip the deployment files for download.

In [ ]:
!zip -r /content/railway_defect_deploy.zip /content/drive/MyDrive/railway_defect_deploy
from google.colab import files
files.download('/content/railway_defect_deploy.zip')
print("Download the zip file and extract:")
print("  - best.pt -> webapp/model/best.pt")
print("  - best.onnx -> webapp/model/best.onnx")
print("  - class_names.json -> webapp/model/class_names.json")